In [2]:
import mesa

class SchellingAgent(mesa.Agent):

    def __init__(self, model, agent_type: int):
        super().__init__(model)
        self.type = agent_type   # 0 = majority, 1 = minority

    def step(self):
        # Get all occupied cells in the 8-cell Moore neighborhood
        neighbors = list(
            self.model.grid.iter_neighbors(self.pos, moore=True, include_center=False)
        )

        if not neighbors:
            return   # isolated — stay put

        similar = sum(1 for n in neighbors if n.type == self.type)
        pct_similar = similar / len(neighbors)

        # Unhappy → move to a random empty cell
        if pct_similar < self.model.homophily:
            self.model.grid.move_to_empty(self)

In [3]:
class SchellingModel(mesa.Model):

    def __init__(
        self,
        width=30, height=30,
        density=0.8,
        minority_pct=0.2,
        homophily=0.3,
        seed=None
    ):
        super().__init__(seed=seed)
        self.homophily = homophily

        # A grid where each cell holds at most one agent
        # torus=True means the edges wrap around (like NetLogo's default)
        self.grid = mesa.space.SingleGrid(width, height, torus=True)

        # Place agents — like NetLogo's "ask patches [ if random-float 1 < density [...] ]"
        for _, pos in self.grid.coord_iter():
            if self.random.random() < density:
                agent_type = 1 if self.random.random() < minority_pct else 0
                agent = SchellingAgent(self, agent_type)
                self.grid.place_agent(agent, pos)

In [6]:
class SchellingModel(mesa.Model):

    def __init__(
        self,
        width=30, height=30,
        density=0.8,
        minority_pct=0.2,
        homophily=0.3,
        seed=None
    ):
        super().__init__(seed=seed)
        self.homophily = homophily

        self.grid = mesa.space.SingleGrid(width, height, torus=True)

        for _, pos in self.grid.coord_iter():
            if self.random.random() < density:
                agent_type = 1 if self.random.random() < minority_pct else 0
                agent = SchellingAgent(self, agent_type)
                self.grid.place_agent(agent, pos)

        self.datacollector = mesa.DataCollector(
            model_reporters={"pct_happy": lambda m: m._pct_happy()},
            agent_reporters={
                "type": "type",
                "x": lambda a: a.pos[0],
                "y": lambda a: a.pos[1],
            }
        )

    # ← _pct_happy and step are class methods, not inside __init__

    def _pct_happy(self):
        happy = 0
        for agent in self.agents:
            nbrs = list(self.grid.iter_neighbors(agent.pos, moore=True, include_center=False))
            if not nbrs:
                happy += 1
                continue
            similar = sum(1 for n in nbrs if n.type == agent.type)
            if similar / len(nbrs) >= self.homophily:
                happy += 1
        return happy / len(self.agents) if self.agents else 1.0

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")